# AraBERT fine-tuning — Track 1 final project (dentiligence)

**Run this on Colab or Kaggle with a free GPU runtime** (`Runtime > Change runtime type > GPU`). This environment has no GPU and no HuggingFace Hub access, so this notebook cannot be executed here — it is written to be correct and complete, then run externally. The output is a checkpoint folder (`models/arabert-sentiment/`) that drops directly into `ArabertSentimentClassifier.load()` in the main repo — no other code changes needed.

**Steps:** install deps -> load the Kaggle Arabic-reviews dataset -> fine-tune AraBERT across a 5-run lr/batch_size sweep, logged to MLflow -> register the best run -> distill 12->6 layers -> quantize to ONNX INT8 -> benchmark the whole journey (Module 4's journey table).

In [ ]:
!pip install -q transformers datasets accelerate mlflow evaluate onnx onnxruntime optimum[onnxruntime] scikit-learn

## 1. Load the dataset

Download an Arabic e-commerce/product review dataset from Kaggle, e.g.
`kaggle datasets download -d abedkhooli/arabic-100k-reviews` (needs a `kaggle.json` API token - see https://www.kaggle.com/docs/api). Upload the CSV to Colab and point `CSV_PATH` at it, or use `kagglehub`/`kaggle` CLI directly in Colab.

In [ ]:
import pandas as pd

CSV_PATH = 'arabic_reviews.csv'  # upload via Colab's file browser, or pull with the kaggle CLI
raw = pd.read_csv(CSV_PATH)

# Same column-detection + rating->label mapping as src/arasent/data.py, kept standalone
# here so this notebook has no dependency on the local package.
TEXT_CANDIDATES = ['text', 'review', 'review_text', 'Review']
LABEL_CANDIDATES = ['label', 'rating', 'sentiment', 'Rating']
text_col = next(c for c in TEXT_CANDIDATES if c in raw.columns)
label_col = next(c for c in LABEL_CANDIDATES if c in raw.columns)

df = raw[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'})

def rating_to_label(r):
    if r <= 2: return 0  # negative
    if r == 3: return 1  # neutral
    return 2              # positive

if df['label'].max() > 2:
    df['label'] = df['label'].apply(rating_to_label)

def clean_text(t):
    t = str(t).strip().replace('ـ', '')
    for a in ['أ', 'إ', 'آ']:
        t = t.replace(a, 'ا')
    return t.replace('ى', 'ي').replace('ة', 'ه')

df['text'] = df['text'].map(clean_text)
df = df[df['text'].str.len() > 0].dropna().reset_index(drop=True)
print(df['label'].value_counts())
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
len(train_df), len(val_df)

## 2. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'aubmindlab/bert-base-arabertv02'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(frame):
    ds = Dataset.from_pandas(frame[['text', 'label']].reset_index(drop=True))
    return ds.map(
        lambda batch: tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128),
        batched=True,
    )

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)

## 3. Fine-tune - a 5-run lr / batch_size sweep, logged to MLflow

Same pattern as `src/arasent/train.py`'s baseline sweep: every run gets its own MLflow run with params, metrics and the git commit tag, so the comparison view in the MLflow UI answers "which hyperparameters produced the best f1_macro".

In [ ]:
import numpy as np
import mlflow
import evaluate
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

mlflow.set_tracking_uri('sqlite:///mlflow_arabert.db')
mlflow.set_experiment('arabert-sentiment')

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1_macro': f1_metric.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

SWEEP = [
    {'lr': 5e-5, 'batch_size': 16},
    {'lr': 3e-5, 'batch_size': 16},
    {'lr': 2e-5, 'batch_size': 32},
    {'lr': 2e-5, 'batch_size': 16},
    {'lr': 1e-5, 'batch_size': 32},
]

best = {'f1_macro': -1, 'run_id': None, 'trainer': None}

for i, cfg in enumerate(SWEEP):
    with mlflow.start_run(run_name=f'arabert-sweep-{i}') as run:
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
        args = TrainingArguments(
            output_dir=f'./arabert-run-{i}',
            learning_rate=cfg['lr'],
            per_device_train_batch_size=cfg['batch_size'],
            per_device_eval_batch_size=32,
            num_train_epochs=3,
            eval_strategy='epoch',
            save_strategy='no',
            logging_steps=50,
            report_to=[],
        )
        trainer = Trainer(
            model=model, args=args,
            train_dataset=train_ds, eval_dataset=val_ds,
            compute_metrics=compute_metrics,
        )
        trainer.train()
        metrics = trainer.evaluate()

        mlflow.log_params({**cfg, 'model_name': MODEL_NAME, 'max_seq_length': 128})
        mlflow.log_metrics({'accuracy': metrics['eval_accuracy'], 'f1_macro': metrics['eval_f1_macro']})

        print(f"run {i}: lr={cfg['lr']} batch_size={cfg['batch_size']} "
              f"f1_macro={metrics['eval_f1_macro']:.4f}")

        if metrics['eval_f1_macro'] > best['f1_macro']:
            best = {'f1_macro': metrics['eval_f1_macro'], 'run_id': run.info.run_id, 'trainer': trainer}

print('Best run:', best['run_id'], 'f1_macro=', best['f1_macro'])

In [ ]:
# Save the best checkpoint - this folder is what goes into the main repo's
# models/arabert-sentiment/, consumed by ArabertSentimentClassifier.load().
OUTPUT_DIR = './arabert-sentiment-best'
best['trainer'].save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR} - zip this folder and copy it to models/arabert-sentiment/ in the repo.')

## 4. Distillation - 12 layers -> 6 layers (Module 4)

Teacher: the fine-tuned AraBERT above (12 layers). Student: a 6-layer BERT initialized from every other teacher layer, then trained with the standard Hinton distillation loss (soft-label KL + hard-label cross-entropy, temperature-scaled).

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoConfig

teacher = best['trainer'].model
teacher.eval()

# Student: half the layers, initialized from every-other teacher layer (a cheap, standard
# initialization trick - much faster to converge than training the student from scratch).
student_config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=3, num_hidden_layers=6)
student = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=student_config, num_labels=3, ignore_mismatched_sizes=True
)

def distillation_loss(student_logits, teacher_logits, labels, T=3.0, alpha=0.7):
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=-1),
        F.softmax(teacher_logits / T, dim=-1),
        reduction='batchmean',
    ) * (T ** 2)
    hard_loss = F.cross_entropy(student_logits, labels)
    return alpha * soft_loss + (1 - alpha) * hard_loss

class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = teacher(**inputs)
        loss = distillation_loss(student_outputs.logits, teacher_outputs.logits, labels)
        return (loss, student_outputs) if return_outputs else loss

distill_args = TrainingArguments(
    output_dir='./arabert-student',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy='epoch',
    save_strategy='no',
    report_to=[],
)
distill_trainer = DistillationTrainer(
    model=student, args=distill_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
distill_trainer.train()
student_metrics = distill_trainer.evaluate()
print('Student (6-layer) metrics:', student_metrics)

STUDENT_DIR = './arabert-sentiment-student-6layer'
distill_trainer.save_model(STUDENT_DIR)
tokenizer.save_pretrained(STUDENT_DIR)

## 5. ONNX export + INT8 quantization, and the journey table (Module 4)

Exports the 6-layer student to ONNX, applies dynamic INT8 quantization via Optimum, and times all three variants (teacher, student, student+INT8) on the same validation rows - the benchmark harness the handbook asks for, done properly: warmup discarded, >=50 timed iterations, same eval set throughout.

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ort_model = ORTModelForSequenceClassification.from_pretrained(STUDENT_DIR, export=True)
ort_model.save_pretrained('./arabert-student-onnx')

quantizer = ORTQuantizer.from_pretrained('./arabert-student-onnx')
qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
quantizer.quantize(save_dir='./arabert-student-onnx-int8', quantization_config=qconfig)
print('Quantized model saved to ./arabert-student-onnx-int8')

In [ ]:
import time
import os

def benchmark(predict_fn, texts, n_warmup=10, n_timed=50):
    for _ in range(n_warmup):
        predict_fn(texts[:1])
    times = []
    for _ in range(n_timed):
        start = time.perf_counter()
        predict_fn(texts[:1])
        times.append((time.perf_counter() - start) * 1000)
    times.sort()
    return {'mean_ms': sum(times) / len(times), 'p95_ms': times[int(0.95 * len(times)) - 1]}

sample_texts = val_df['text'].tolist()[:100]

def teacher_predict(texts):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        return teacher(**enc).logits

def student_predict(texts):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        return student(**enc).logits

ort_int8_model = ORTModelForSequenceClassification.from_pretrained('./arabert-student-onnx-int8')

def onnx_int8_predict(texts):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
    return ort_int8_model(**enc).logits

def dir_size_mb(path):
    total = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(path) for f in fs)
    return round(total / 1e6, 1)

journey = [
    {'variant': 'Teacher (12-layer AraBERT)', **benchmark(teacher_predict, sample_texts),
     'size_mb': dir_size_mb(OUTPUT_DIR), 'f1_macro': best['f1_macro']},
    {'variant': 'Student (6-layer, distilled)', **benchmark(student_predict, sample_texts),
     'size_mb': dir_size_mb(STUDENT_DIR), 'f1_macro': student_metrics['eval_f1_macro']},
    {'variant': 'Student + ONNX INT8', **benchmark(onnx_int8_predict, sample_texts),
     'size_mb': dir_size_mb('./arabert-student-onnx-int8'), 'f1_macro': None},
]

journey_df = pd.DataFrame(journey)
journey_df.to_csv('journey_table.csv', index=False)
journey_df

## 6. Ship it back to the main repo

1. Zip `./arabert-student-onnx-int8/` (or the plain `STUDENT_DIR` PyTorch checkpoint, if you'd    rather serve via `transformers` than ONNX Runtime) and download it from Colab.
2. Unzip into `models/arabert-sentiment/` in the main repo.
3. Set `ARASENT_MODEL_BACKEND=transformer` and restart the API - `predict.py` picks it up with    no other code changes.
4. Copy `journey_table.csv` into `reports/module-4-journey.csv` for the optimization report.